# Metaheuristics for Multi-Resource Workload Allocation

## Problem Definition

This coursework investigates a multi-resource model selection problem.
Each candidate model has a utility value and consumes several resources.
The objective is to select a subset of models that maximises total utility
without exceeding any resource capacity.

A solution is represented by a binary vector: 1 means a model is selected,
and 0 means it is not selected.

## 1. Data loading

NumPy is used to store and calculate model utilities and resource demands.
The loading function follows the structure of the lecturer-provided loader,
with an additional check on the number of data entries.

The file contains the number of models (n), the number of resources (m),
n utility values, m × n resource demands, and m capacity limits, in that order.
The resource demand matrix has shape (m, n): each row represents a resource,
and each column represents a model.

The two lecturer-provided data files are included alongside this
notebook. Keep both files in the same directory as the notebook
and run the notebook from that directory.

In [1]:
import numpy as np


def load_workload_instance(file_path):
    """Read model utilities, resource demands and capacities."""
    with open(file_path, "r", encoding="utf-8") as f:
        tokens = f.read().split()

    n = int(tokens[0])
    m = int(tokens[1])

    expected_count = 2 + n + m * n + m
    if len(tokens) != expected_count:
        raise ValueError("The data quantity is incorrect. Please check the data file.")

    idx = 2

    utilities = np.array(
        tokens[idx:idx + n], dtype=float
    )
    idx += n

    demands = np.array(
        tokens[idx:idx + m * n], dtype=float
    ).reshape(m, n)
    idx += m * n

    capacities = np.array(
        tokens[idx:idx + m], dtype=float
    )

    return n, m, utilities, demands, capacities

## 2. Instance 1: Data Summary

Instance 1 contains 28 candidate models and two resource constraints,
each with a capacity of 500. It is used to develop and evaluate the
baseline hill-climbing algorithm in Part 1.

The following output reports the instance dimensions and capacity limits
to check that they match the assignment specification.

In [4]:
n, m, utilities, demands, capacities = load_workload_instance(
   "multi_resource_n28_m2.txt"
)

print("Number of models:", n)
print("Number of resources:", m)
print("Resource capacities:", capacities)

Number of models: 28
Number of resources: 2
Resource capacities: [500. 500.]


## 3. Solution Representation and Evaluation

A solution is represented by a binary vector of length n.
A value of 1 selects a model, while 0 excludes it.

The total utility and resource usage are calculated by summing
the contributions of the selected models. A solution is feasible
only if every resource usage is within its capacity.

Selecting models 1 and 3 gives a total utility of 24,405 and resource usage of 130 and 155. Both resource usages are below their respective capacities of 500, so the solution is feasible.

This manually constructed example checks the evaluation calculation. It is not a claim that the solution is the best.

In [6]:
def evaluate_solution(x, utilities, demands, capacities):
    """Return total utility, resource usage and feasibility."""
    total_utility = utilities @ x
    resource_usage = demands @ x
    feasible = bool(np.all(resource_usage <= capacities))

    return total_utility, resource_usage, feasible


# Start with all models unselected.
x = np.zeros(n, dtype=int)

# Select model 1 and model 3.
x[0] = 1
x[2] = 1

value, usage, feasible = evaluate_solution(
    x, utilities, demands, capacities
)

print("Selection vector:", x)
print("Total utility:", value)
print("Resource usage:", usage)
print("Feasible:", feasible)

Selection vector: [1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Total utility: 24405.0
Resource usage: [130. 155.]
Feasible: True


### Feasibility Check

To check the constraint evaluation, all 28 models are selected.
This deliberately creates an infeasible solution, allowing us
to verify that the evaluation function detects resource violations.

In [7]:
# Select all 28 models.
x_all = np.ones(n, dtype=int)

value_all, usage_all, feasible_all = evaluate_solution(
    x_all, utilities, demands, capacities
)

print("Resource usage:", usage_all)
print("Resource capacities:", capacities)
print("Feasible:", feasible_all)

Resource usage: [1125.  995.]
Resource capacities: [500. 500.]
Feasible: False


Selecting all 28 models uses 1,125 units of resource 1 and 995 units of resource 2. Both exceed their capacities of 500, so the evaluation function correctly identifies this solution as infeasible.